# Nemotron v1 diagnostics

Checks the fine-tuned Nemotron v1 without training. It uses the same inference and scoring as `notebooks/fine-tune/nemotron-3.5-asr-streaming-0.6b.ipynb`.

1. **Base model in NeMo** on the test set (`hi-IN` and `auto`): the correct "before" numbers for the same pipeline as v1.
2. **Base model and v1 on the validation set**: empty outputs, and the leaked and other utterances scored separately.
3. **Empty v1 test outputs** (from the saved `hyps-560ms.jsonl`): their durations, raw outputs and audio.

**Before you run:**
- Copy `data/interim/openslr104_validation_leaked.csv` to the same path in the project folder on Drive.
- Check the Drive paths in the Settings cell.
- An L4 GPU is enough. Cells 7 and 8 take most of the time (about 10–15 minutes).

## Settings

In [ ]:
PROJECT = "/content/drive/MyDrive/projects/asr"
MODEL = "nvidia/nemotron-3.5-asr-streaming-0.6b"
PROMPT = "hi-IN"

SPLITS = ("validation", "test")
ZIPS = {split: f"{PROJECT}/openslr104_{split}.zip" for split in SPLITS}
AUDIO_ROOT = "/content/openslr104"
V1_BEST = f"{PROJECT}/checkpoints/nemotron-3.5-asr-streaming-0.6b-v1/best.nemo"
V1_RESULTS = f"{PROJECT}/results/nemotron-3.5-asr-streaming-0.6b-finetuned"

CONTEXT = {560: [56, 6], 1120: [56, 13]}  # latency (ms) -> att_context_size
INFER_BATCH = 32

## Install

If Colab asks to restart the session after the install, restart, then run the Settings cell again and continue from the next cell.

In [ ]:
!apt-get install -y -qq libsndfile1 ffmpeg
!pip install -q Cython packaging
!pip install -q "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@main" jiwer

## Setup

In [ ]:
import gc
import json
import os
import re
import shutil
import sys
import zipfile
from pathlib import Path

import pandas as pd
import torch
from google.colab import drive
from IPython.display import Audio, display
from tqdm.auto import tqdm

drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, f"{PROJECT}/src")

from hinglish_asr.metrics import score
from hinglish_asr.utils import Normalizer
from nemo.collections.asr.models import ASRModel

print(torch.cuda.get_device_name(0))

## Audio

The zips are copied to the local disk and unzipped there. A split that is already unzipped in this session is skipped.

In [ ]:
for split, zip_path in ZIPS.items():
    target = Path(AUDIO_ROOT) / split
    if target.exists():
        continue
    local_zip = Path("/content") / Path(zip_path).name
    shutil.copy(zip_path, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        for member in tqdm(z.infolist(), desc=f"unzip {split}"):
            z.extract(member, target)
    local_zip.unlink()

## Manifests

Each clip is found by file name, because the manifests have laptop paths.

In [ ]:
rows, paths = {}, {}
for split in SPLITS:
    wavs = {p.name: str(p) for p in (Path(AUDIO_ROOT) / split).rglob("*.wav")}
    with open(f"data/processed/openslr104/{split}_manifest.jsonl", encoding="utf-8") as f:
        rows[split] = [json.loads(line) for line in f]
    missing = [r["utt_id"] for r in rows[split] if Path(r["audio_path"]).name not in wavs]
    assert not missing, f"{split}: {len(missing)} clips missing, e.g. {missing[:3]}"
    paths[split] = [wavs[Path(r["audio_path"]).name] for r in rows[split]]
    print(f"{split}: {len(rows[split])} clips")
refs = {split: [r["text"] for r in rows[split]] for split in SPLITS}

leaked_ids = set(pd.read_csv("data/interim/openslr104_validation_leaked.csv")["utt_id"])
is_leak = [r["utt_id"] in leaked_ids for r in rows["validation"]]
print(f"{sum(is_leak)} leaked validation clips (expected 185)")

## Scoring and inference

`transcribe` is the same as in the fine-tune notebook: whole clips, with the encoder attention limited to one streaming latency. `strip=False` keeps the language tag, to see the raw output. `score_groups` scores the validation set in total, and the leaked and other utterances separately.

In [ ]:
norm = Normalizer("data/lists/loanwords.csv", "data/lists/transliterations.csv")
assert score(refs["test"], refs["test"], norm) == {"wer": 0.0, "t_wer": 0.0}
LANG_TAG = re.compile(r"\s*<[a-z]{2}-[A-Z]{2}>")


def transcribe(model, audio_paths, ms, language=PROMPT, strip=True):
    default = list(model.encoder.att_context_size)
    model.encoder.set_default_att_context_size(att_context_size=CONTEXT[ms])
    try:
        out = model.transcribe(audio_paths, batch_size=INFER_BATCH, target_lang=language, verbose=True)
    finally:
        model.encoder.set_default_att_context_size(att_context_size=default)
    out = out[0] if isinstance(out, tuple) else out
    texts = [h.text if hasattr(h, "text") else h for h in out]
    return [LANG_TAG.sub("", t).strip() for t in texts] if strip else texts


def score_groups(hyps):
    out = {"all": score(refs["validation"], hyps, norm), "empty": sum(not h for h in hyps)}
    for name, keep in (("leaked", True), ("other", False)):
        r = [x for x, leak in zip(refs["validation"], is_leak) if leak == keep]
        h = [x for x, leak in zip(hyps, is_leak) if leak == keep]
        out[name] = score(r, h, norm)
    return out

## 1. Base model in NeMo

- `hi-IN` on the test set: the correct "before" number for v1 (v1 test WER at 560 ms: 0.392).
- `auto` on the test set: must be close to the transformers zero-shot result (WER 0.605, T-WER 0.403).
- Validation: the "before" numbers for cell 2.

In [ ]:
model = ASRModel.from_pretrained(MODEL, map_location="cuda")

for lang in ("hi-IN", "auto"):
    hyps = transcribe(model, paths["test"], 560, language=lang)
    print(f"base, test, 560 ms, {lang}: {score(refs['test'], hyps, norm)}, {sum(not h for h in hyps)} empty")

print("base, validation, 560 ms:", score_groups(transcribe(model, paths["validation"], 560)))

## 2. v1 on the validation set

- **empty:** many empty outputs here mean that the fine-tune causes them everywhere. Few mean that the problem is in the test data.
- **leaked against other:** how much the leaked utterances help the validation score.

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

v1 = ASRModel.restore_from(V1_BEST, map_location="cuda")
print("v1, validation, 560 ms:", score_groups(transcribe(v1, paths["validation"], 560)))

## 3. Empty v1 test outputs

Read from the saved v1 outputs, with no new run. Compares the durations of the empty clips with all clips, and saves the list for listening.

In [ ]:
test_hyps = pd.read_json(f"{V1_RESULTS}/hyps-560ms.jsonl", lines=True, dtype={"utt_id": str, "ref": str, "hyp": str})
duration = {r["utt_id"]: r["duration"] for r in rows["test"]}
test_hyps["duration"] = test_hyps["utt_id"].map(duration)
empty = test_hyps[test_hyps["hyp"].fillna("").str.strip() == ""]

print(f"{len(empty)} empty of {len(test_hyps)} test clips (expected 220)")
print(pd.DataFrame({"empty": empty["duration"].describe(), "all": test_hyps["duration"].describe()}).round(1))
empty[["utt_id", "duration", "ref"]].to_csv(f"{V1_RESULTS}/empty-560ms.csv", index=False)

### Raw output and audio for 10 empty clips

The raw output keeps the language tag. If it shows only a tag, the model output no words. Listen to the first 3 clips and compare them with the reference.

In [ ]:
path_by_id = dict(zip((r["utt_id"] for r in rows["test"]), paths["test"]))
sample = empty["utt_id"].head(10).tolist()
raw = transcribe(v1, [path_by_id[u] for u in sample], 560, strip=False)

for i, (u, h) in enumerate(zip(sample, raw)):
    print(u, "| raw:", repr(h))
    print("   ref:", empty.loc[empty["utt_id"] == u, "ref"].item())
    if i < 3:
        display(Audio(path_by_id[u]))

## How to read the results

| Result | Question it answers |
|---|---|
| Base `hi-IN` test WER against v1 test WER (0.392) | The real gain from the fine-tune, with the same pipeline |
| Base `auto` test WER against transformers (0.605) | Do NeMo and transformers give the same score? |
| Empty count on validation, base against v1 | Did the fine-tune cause empty outputs everywhere, or only on the test set? |
| "leaked" against "other" for v1 | How much the leak helps the validation score |
| Durations of empty clips against all clips | Are the empty clips very short or very long? |
| Raw outputs | Is the output really empty, or only a language tag? |
| Audio | Does the audio match the reference? |